# Tugas Besar 2 IF3170 Inteligensi Artifisial

Notebook pipeline data dan pemodelan untuk kompetisi Kaggle `T2-IF3170-2026-2027`.

---

## Group 30

- Ega Luthfi Rais (13524115)
- Faris Wirakusuma Triawan (13524130)
- Ahmad Fauzan Putra (13524141)
- Jonathan Harijadi (13524144)
- Muhammad Hartawan Haidir (13524147)

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding (Optional)**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Preparation Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)


---

# Introduction <a name="1"></a>

---

This section frames the problem in the words of the group.

## Overview

*To be written by the group.*

## Aim

*To be written by the group.*

## Metric

*To be written by the group.*

## Dataset

*To be written by the group.*

## Approach

*To be written by the group.*

---

# Initialization <a name="2"></a>

---

This section prepares the environment, the settings, and the data files.

## Environment Setup

The following cell lists the base libraries. The install line is commented by default.

In [ ]:
# %pip install -q numpy pandas

## Import Libraries

Every import of the notebook is kept in this cell.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

## Settings

Paths and shared constants are kept in this class.

In [ ]:
class Settings:
    SEED = int(os.environ.get("SEED", 42))
    DATA_DIR = ROOT / "data" / "raw"
    PREP_DIR = ROOT / "output" / "prepared"
    OUTPUT_DIR = ROOT / "output"
    TARGET = "is_canceled"
    ID_COL = "id"

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Seed Everything

A fixed seed keeps the results reproducible.

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

## Load Dataset

The competition files are read here. The `id` column stays as text because some values resemble numbers.

In [ ]:
train_df = pd.read_csv(CFG.DATA_DIR / 'train.csv', dtype={'id': str})
test_df  = pd.read_csv(CFG.DATA_DIR / 'test.csv', dtype={'id': str})

print(f'Train: {train_df.shape}  Test: {test_df.shape}')

---

# Data Understanding (Optional) <a name="3"></a>

---

Data understanding builds the knowledge that later preparation decisions rest on.

## Describe Data

This part covers the size, the attribute types, and the meaning of each column.

In [ ]:
# Write your code here

## Explore Data

This part covers distributions and relationships between attributes.

In [ ]:
# Write your code here

## Verify Data Quality

This part covers completeness, consistency, and the validity of values.

In [ ]:
# Write your code here

---

# Training and Validation Split <a name="4"></a>

---

The split comes before data preparation so that validation data stays unseen. Only `train.csv` is split, and `test.csv` serves the Kaggle submission alone.

In [ ]:
# train_set, val_set = ...

---

# Data Preparation <a name="5"></a>

---

Each process below applies where the data calls for it, and the Insights block under it records the reason for the decision.

Preparation steps may use `sklearn`. Every statistic comes from the training split only, then transforms the other sets.

## Select Data

Selection decides which records and attributes stay in the analysis.

In [ ]:
# Write your code here

## Clean Data

Cleaning raises data quality to the level the chosen models need.

### Missing Values

This part covers missing values, if any.

In [ ]:
# Write your code here

### Duplicates

This part covers repeated records.

In [ ]:
# Write your code here

### Outliers

This part covers values far from the rest of the data.

In [ ]:
# Write your code here

### Data Errors

This part covers wrong types, inconsistent labels, and impossible values.

In [ ]:
# Write your code here

## Construct Data

Construction derives attributes and transforms values for modeling.

### Attribute Construction

This part covers new attributes derived from existing ones.

In [ ]:
# Write your code here

### Feature Scaling

This part covers normalization to the range 0 to 1 and standardization.

In [ ]:
# Write your code here

### Encoding

This part covers the numeric representation of categorical attributes.

In [ ]:
# Write your code here

### Dimensionality Reduction

This part covers fewer dimensions that keep the meaning of the data.

In [ ]:
# Write your code here

## Class Balance

This part covers the proportion of each class in the training data.

In [ ]:
# Write your code here

## Format Data

Formatting produces the structure the models read, such as a numeric matrix.

In [ ]:
# Write your code here

---

# Preparation Pipeline <a name="6"></a>

---

The pipeline combines the preparation steps so they run in one consistent order.

In [ ]:
# Write your code here

---

# Modeling and Validation <a name="7"></a>

---

The target is `is_canceled`. The models here are implemented from scratch, with math libraries such as NumPy.

### Catatan implementasi

Kelima classifier (KNN, DTL, Logistic Regression, SVM, Naive Bayes) di `src/models/` diimplementasikan from scratch dengan NumPy; pandas hanya untuk I/O dan penanganan tabel. scikit-learn dipakai untuk preprocessing di `DataPreprocessor` (`src/data/preprocessing.py`) dan sebagai pembanding, bukan untuk model submission.

## K-Nearest Neighbors

This part covers the nearest neighbor classifier and its validation.

In [ ]:
import json
from sklearn.metrics import f1_score, classification_report        
from sklearn.neighbors import KNeighborsClassifier
from src.models.knn import KNN

tr, va = np.load(CFG.PREP_DIR / "train.npz"), np.load(CFG.PREP_DIR / "val.npz")
X_train, y_train, X_val, y_val = tr["X"], tr["y"], va["X"], va["y"]

ks = [1, 3, 5, 7, 9, 11, 15, 21, 31, 51]
probe = KNN(k=max(ks), p=2, weights="distance").fit(X_train, y_train)
dists, idx = probe.kneighbors(X_val)                 
labels = y_train[idx]
sweep = {"uniform": [], "distance": []}
for k in ks:
    sweep["uniform"].append(f1_score(y_val, (labels[:, :k].mean(1) >= .5).astype(int), average="macro"))
    w = 1 / (dists[:, :k] + 1e-12)
    sweep["distance"].append(f1_score(y_val, (((w * labels[:, :k]).sum(1) / w.sum(1)) >= .5).astype(int), average="macro"))
best_w = max(sweep, key=lambda s: max(sweep[s]))
best_k = ks[int(np.argmax(sweep[best_w]))]
print("k terbaik:", best_k, "| bobot:", best_w, "| macro F1:", round(max(sweep[best_w]), 4))

# model final + jarak tetangga validasi + pembanding scikit-learn
knn = KNN(k=best_k, p=2, weights=best_w).fit(X_train, y_train)
pred_knn = knn.predict(X_val)
val_neighbor_dists = knn.val_neighbor_dists            # (n_val, k) jarak ke tetangga terdekat
print("rata-rata jarak tetangga ke-1 / ke-k:", val_neighbor_dists[:, 0].mean().round(3), val_neighbor_dists[:, -1].mean().round(3))
print(classification_report(y_val, pred_knn, digits=4))

knn_sk = KNeighborsClassifier(n_neighbors=best_k, weights=best_w, p=2, algorithm="brute").fit(X_train, y_train)
print("macro F1 scratch :", round(f1_score(y_val, pred_knn, average="macro"), 4))
print("macro F1 sklearn :", round(f1_score(y_val, knn_sk.predict(X_val), average="macro"), 4))
print("prediksi sama    :", (pred_knn == knn_sk.predict(X_val)).mean())

## Decision Tree Learning

This part covers one chosen variant, which is ID3, C4.5, or CART, and its validation.

In [ ]:
import sys
from pathlib import Path
import numpy as np
from sklearn.metrics import classification_report, f1_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.decision_tree import DecisionTree

if "X_train" not in globals():
    _train = np.load(ROOT / "output" / "prepared" / "train.npz")
    _val = np.load(ROOT / "output" / "prepared" / "val.npz")
    X_train, y_train = _train["X"], _train["y"]
    X_val, y_val = _val["X"], _val["y"]

dt_model = DecisionTree(criterion="entropy", max_depth=12, min_samples_leaf=5)
dt_model.fit(X_train, y_train)
dt_val_pred = dt_model.predict(X_val)

print(f"DecisionTree (ID3) val macro F1: {f1_score(y_val, dt_val_pred, average='macro'):.4f}")
print(classification_report(y_val, dt_val_pred, digits=4))

## Logistic Regression

This part covers the model with L1, L2, and Elastic Net regularization.

In [ ]:
# Write your code here

## Support Vector Machine

This part covers the linear, polynomial, and Gaussian kernels.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Find the project root so `src.models` is importable from any notebook dir.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "models" / "svm.py").exists())
sys.path.insert(0, str(ROOT))

from src.models.svm import SVM

train = np.load(ROOT / "output" / "prepared" / "train.npz")
val = np.load(ROOT / "output" / "prepared" / "val.npz")
X_train, y_train = train["X"], train["y"]
X_val, y_val = val["X"], val["y"]

# Linear SVM uses all rows; the two kernel SVMs subsample via max_samples.
kernel_params = {
    "linear": dict(kernel="linear", C=1.0, learning_rate=0.002, epochs=50),
    "polynomial": dict(kernel="poly", degree=3, coef0=1.0, gamma=0.1, C=1.0,
                       learning_rate=0.05, epochs=30, max_samples=2000),
    "gaussian": dict(kernel="gaussian", gamma=0.15, C=1.0,
                     learning_rate=0.05, epochs=30, max_samples=2000),
}

svm_rows, svm_models = [], {}
for name, params in kernel_params.items():
    model = SVM(**params)
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_s = time.perf_counter() - t0
    t0 = time.perf_counter()
    pred = model.predict(X_val)
    predict_s = time.perf_counter() - t0
    score = f1_score(y_val, pred, average="macro")
    svm_rows.append({"kernel": name, "fit_s": fit_s, "predict_s": predict_s, "macro_f1": score})
    svm_models[name] = model
    print(f"{name:<10} fit={fit_s:6.2f}s  predict={predict_s:6.2f}s  macro F1={score:.4f}")

svm_table = pd.DataFrame(svm_rows).sort_values("macro_f1", ascending=False)
display(svm_table)

best_kernel = svm_table.iloc[0]["kernel"]
best_pred = svm_models[best_kernel].predict(X_val)
print(classification_report(y_val, best_pred, digits=4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(svm_table["kernel"], svm_table["macro_f1"], color="#4C72B0")
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("macro F1")
axes[0].set_title("SVM validation per kernel")

labels = np.unique(y_val)
matrix = confusion_matrix(y_val, best_pred)
axes[1].imshow(matrix, cmap="Blues")
axes[1].set_xticks(range(len(labels)), labels)
axes[1].set_yticks(range(len(labels)), labels)
axes[1].set_xlabel("predicted")
axes[1].set_ylabel("actual")
axes[1].set_title(f"Confusion matrix — {best_kernel}")
for i in range(len(labels)):
    for j in range(len(labels)):
        axes[1].text(j, i, matrix[i, j], ha="center", va="center",
                     color="white" if matrix[i, j] > matrix.max() / 2 else "black")

plt.tight_layout()
plt.show()

## Naive Bayes

This part covers the Naive Bayes classifier and its validation.

In [ ]:
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from src.models.naive_bayes import NaiveBayes, CategoricalNaiveBayes, backward_elimination

gnb = NaiveBayes().fit(X_train, y_train)
print("Gaussian scratch :", round(f1_score(y_val, gnb.predict(X_val), average="macro"), 4),
      "| sklearn:", round(f1_score(y_val, GaussianNB().fit(X_train, y_train).predict(X_val), average="macro"), 4))
cnb_all = CategoricalNaiveBayes(n_bins=20).fit(X_train, y_train)
print("Kategorikal (semua fitur):", round(f1_score(y_val, cnb_all.predict(X_val), average="macro"), 4))

# seleksi fitur backward (CV di train, ~2-3 menit) + pembanding
feature_names = json.load(open(CFG.PREP_DIR / "feature_names.json"))
cols, history = backward_elimination(X_train, y_train, lambda: CategoricalNaiveBayes(n_bins=20))
print("dibuang:", [feature_names[j] for j, _ in history])

nb = CategoricalNaiveBayes(n_bins=20).fit(X_train[:, cols], y_train)
pred_nb = nb.predict(X_val[:, cols])
print(classification_report(y_val, pred_nb, digits=4))

B_tr, B_va = nb._digitize(X_train[:, cols]), nb._digitize(X_val[:, cols])
nb_sk = CategoricalNB(alpha=1.0, min_categories=nb.n_categories_).fit(B_tr, y_train)
print("macro F1 scratch :", round(f1_score(y_val, pred_nb, average="macro"), 4))
print("macro F1 sklearn :", round(f1_score(y_val, nb_sk.predict(B_va), average="macro"), 4))

## Model Evaluation

This part compares the validation results of all models.

In [ ]:
# Write your code here

## Improvements (Optional)

This part covers hyperparameter tuning, cross-validation, and further visualization.

In [ ]:
# Write your code here

---

# Comparison with scikit-learn <a name="8"></a>

---

This section compares each from-scratch model with its `scikit-learn` counterpart.

Model scikit-learn di bagian ini **hanya pembanding**. Submission akhir tetap memakai model from scratch terbaik dari validation.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_sklearn = DecisionTreeClassifier(
    criterion="entropy", max_depth=12, min_samples_leaf=5, random_state=42
)
dt_sklearn.fit(X_train, y_train)
dt_sklearn_pred = dt_sklearn.predict(X_val)

print(f"DTL Scratch (ID3) val macro F1: {f1_score(y_val, dt_val_pred, average='macro'):.4f}")
print(f"DTL Sklearn (ID3) val macro F1: {f1_score(y_val, dt_sklearn_pred, average='macro'):.4f}")
print(f"Prediction agreement          : {(dt_val_pred == dt_sklearn_pred).mean():.4f}")

---

# Saving and Loading Models <a name="9"></a>

---

This section stores the trained models and restores them for prediction.

In [ ]:
dt_path = ROOT / "output" / "decision_tree.pkl"
dt_model.save(dt_path)
dt_loaded = DecisionTree.load(dt_path)

assert np.array_equal(dt_loaded.predict(X_val), dt_val_pred)
print(f"Saved and verified {dt_path}")

---

# Kaggle Submission <a name="10"></a>

---

The final pipeline and model are fitted on the original training set, and the test set is only transformed. The submission file contains the `id` and `is_canceled` columns.

In [ ]:
# Write your code here

---

# Error Analysis <a name="11"></a>

---

The analysis supports each step taken in this notebook.